#Домашнє завдання до тем apply(), groupby(), pivot_table()

В цьому домашньому завданні продовжуємо працювати з набором даних `supermarket_sales.csv`.

0. Імпортуйте бібліотеку pandas та зчитайте дані у змінну `df` типу `pandas.DataFrame`.

In [2]:
import numpy as np
import pandas as pd

In [8]:
data_path = '../data/supermarket_sales_new.csv'
df = pd.read_csv(data_path)
type(df)

pandas.core.frame.DataFrame

1. Дослідимо, який філіал супермаркету ('Branch') є найприбутковішим. Для цього знайдіть сумарний прибуток за кожним філіалом і виявіть, який філіал має найвищий.

In [4]:
branch_profit = df.groupby('Branch')['Total'].sum().round(2)
branch_profit

Branch
A    106200.37
B    106197.67
C    110568.71
Name: Total, dtype: float64

2. В якому місті знайходиться філіал з найвищим прибутком? Може в тому місці нам розмітисти ще один магазин.  
Знайдіть відповідь за допомогою функціоналу Pandas.

In [4]:
best_branch = df.groupby('Branch')['Total'].sum().idxmax()
best_city = df.loc[df['Branch'] == best_branch, 'City'].iloc[0]

best_branch, best_city

('C', 'Naypyitaw')

3.1. Створіть зводну таблицю, яка покаже, скільки покупок (інвойсів) було зроблено в кожній з філій (`Branch`) за різними категоріями товарів. Запишіть таблицю в змінну `invoices_by_category` і виведіть змінну на екран.
Ця таблиця допоможе проаналізувати, в якій філії купують найбільше товарів кожної з категорій.

In [5]:
invoices_by_category = df.pivot_table(
    index='Branch',
    columns='Product line',
    values='Invoice ID',
    aggfunc='count',
    fill_value=0
)

invoices_by_category

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Branch,,,,,,
A,60,51,58,47,65,59
B,55,62,50,53,50,62
C,55,65,66,52,45,45


Очікуваний результат:

![](https://drive.google.com/uc?export=view&id=1rueAdko6S3UxIHGtojetTxlES-EyM6Yb)

3.2. Викристовуючи змінну `invoices_by_category` дайте відповідь програмно (тобто значення треба не просто знайти очима, а вивести за допомогою коду), в якому філіалі магазину (`Branch`) найбільше інвойсів із покупкою товарів категорії "Електронні аксесуари" (`Electronic accessories`)?


In [6]:
branch = invoices_by_category['Electronic accessories'].idxmax()
count = invoices_by_category['Electronic accessories'].max()

branch, count

('A', 60)

4-6. **Творче завдання на розвиток аналітичного мислення**

Крок 1. Сформулюйте ТРИ питання (гіпотези) до наявних даних, які допомогли б вам зрозуміти, які користувачі що, де та коли найбільше/найменше купують, аби дати на основі цих гіпотез рекомендації бізнесу. Звісно питання мають бути не тими, на які ми вже відповіли в завданнях модулю.

Крок 2. Знайдіть відповіді на свої питання з допомогою функціоналу pandas.

Крок 3. Напишіть, як відповідь на це питання може бути використана для прийняття бізнес рішень.   
   
 Питання можуть бути будь-якої складності, але їх має бути 3. Кожне питання оцінюється як 1 завдання. Без виконання цього завдання ДЗ не приймається. Якщо є питання щодо виконання - пишіть у чат 🙌


### Гіпотеза 1: Існують години з мінімальною купівельною активністю
**Питання:**  
Чи є години, у які кількість покупок стабільно найнижча незалежно від філіалу та міста?


In [13]:
hour_pivot = df.pivot_table(
    index=['Branch', 'City'],   
    columns='Hour',
    values='Invoice ID',
    aggfunc='count',
    fill_value=0
)

hour_pivot

,Hour,10,11,12,13,14,15,16,17,18,19,20
Branch,City,,,,,,,,,,,
A,Yangon,38,35,33,31,25,37,32,27,33,27,22
B,Mandalay,26,33,25,38,30,32,17,20,35,50,26
C,Naypyitaw,37,22,31,34,28,33,28,27,25,36,27


In [23]:
total_by_hour = df.groupby('Hour')['Invoice ID'].count()

median_value = total_by_hour.median()
mean_value = total_by_hour.mean()

threshold = mean_value * 0.7      # 3. Знайдемо години, які значно нижче середнього (Наприклад, нижче 70% від середнього)

low_hours = total_by_hour[total_by_hour < threshold]

total_by_hour, low_hours

(Hour
 10    101
 11     90
 12     89
 13    103
 14     83
 15    102
 16     77
 17     74
 18     93
 19    113
 20     75
 Name: Invoice ID, dtype: int64,
 Series([], Name: Invoice ID, dtype: int64))

### Аналіз годин купівельної активності

На основі даних про кількість покупок у різні години доби можна виділити як періоди високої активності, так і години зі значно нижчим трафіком.

#### Години з найвищою активністю
- **19:00 – 113 покупок** — абсолютний пік. Це логічно, оскільки більшість покупців повертаються з роботи та заходять у магазин дорогою додому.
- **13:00 – 103 покупки** — обідній пік.
- **15:00 – 102 покупки** — активний денний період.
- **10:00 – 101 покупка** — стабільний ранковий старт після відкриття магазинів.

Такі результати відповідають типовій поведінці покупців: ранковий підйом, обідній пік і сильний вечірній трафік.

#### Години з найнижчою активністю
- **17:00 – 74 покупки** — найслабша година.
- **20:00 – 75 покупок**
- **16:00 – 77 покупок**
- **14:00 – 83 покупки**

Ці години суттєво нижчі за середнє значення (≈93.6).  
Особливо виділяється **17:00** — глобально найслабша година по всіх філіалах разом.

Причини низької активності:
- **17:00:** більшість людей ще не закінчили робочий день, вечірній пік ще не почався.  
- **20:00:** частина покупців уже вдома, магазини близькі до завершення роботи.  
- **16:00 та 14:00:** спад після обідніх піків.

### Висновок
У супермаркетах існує чітко виражена структура активності:  
ранковий та обідній піки, а також сильний вечірній пік.  
Години між ними мають значно нижчу активність, особливо **17:00**.

### Рекомендації для бізнесу
1. **Оптимізувати графік персоналу** у години з низьким навантаженням (14:00–17:00 та 20:00).
2. **Запускати “вечірні акції”** після 17:00, щоб пом’якшити спад продажів.
3. **Перенести викладку товарів або технічні роботи** на найнижчу годину — 17:00.
4. **Посилити промоактивність у пікові години** (10:00–13:00 та 18:00–19:00), коли покупці найбільш активні.



### Гіпотеза 2: Певні категорії товарів частіше купують у містах з вищим середнім чеком або у вечірні години
**Питання:**  
Чи існують категорії товарів, які частіше купують у містах з вищим середнім чеком або в пікові години (наприклад, після 18:00)?


In [24]:
city_avg = df.groupby('City')['Total'].mean().sort_values(ascending=False)
city_avg

City
Naypyitaw    337.099715
Mandalay     319.872506
Yangon       312.354031
Name: Total, dtype: float64

In [36]:
categories_pivot = df.pivot_table(
    index='City',
    columns='Product line',
    values='Invoice ID',
    aggfunc='count',
    fill_value=0
)

categories_pivot = categories_pivot.div(categories_pivot.sum(axis=1), axis=0).round(3)

categories_pivot

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
City,,,,,,
Mandalay,0.166,0.187,0.151,0.160,0.151,0.187
Naypyitaw,0.168,0.198,0.201,0.159,0.137,0.137
Yangon,0.176,0.150,0.171,0.138,0.191,0.174


In [38]:
df['TimeGroup'] = np.where(df['Hour'] >= 18, 'Evening (>=18)', 'Daytime (<18)')

counts = df.pivot_table(
    index=['City', 'TimeGroup'],      # Місто + період
    columns='Product line',           # Категорія товару
    values='Invoice ID',              # Рахуємо інвойси
    aggfunc='count',
    fill_value=0
)

shares = counts.div(counts.sum(axis=1), axis=0).round(2)

shares

Product line              Electronic accessories  Fashion accessories  \
City      TimeGroup                                                     
Mandalay  Daytime (<18)                     0.14                 0.15   
          Evening (>=18)                    0.21                 0.26   
Naypyitaw Daytime (<18)                     0.17                 0.21   
          Evening (>=18)                    0.16                 0.17   
Yangon    Daytime (<18)                     0.17                 0.16   
          Evening (>=18)                    0.18                 0.12   

Product line              Food and beverages  Health and beauty  \
City      TimeGroup                                               
Mandalay  Daytime (<18)                 0.14               0.19   
          Evening (>=18)                0.17               0.10   
Naypyitaw Daytime (<18)                 0.17               0.15   
          Evening (>=18)                0.30               0.17   
Yangon    Daytime (<18)                 0.16               0.12   
          Evening (>=18)                0.22               0.20   

Product line              Home and lifestyle  Sports and travel  
City      TimeGroup                                              
Mandalay  Daytime (<18)                 0.16               0.21  
          Evening (>=18)                0.13               0.14  
Naypyitaw Daytime (<18)                 0.15               0.15  
          Evening (>=18)                0.11               0.09  
Yangon    Daytime (<18)                 0.21               0.18  
          Evening (>=18)                0.12               0.16

## Висновок

## 1. Міста з вищим середнім чеком

Місто **Naypyitaw** має найвищий середній чек (~337).  
Найбільші частки покупок припадають на:

- **Food and beverages — 0.201**
- **Fashion accessories — 0.198**

Ці категорії є найбільш популярними в місті з найвищою платоспроможністю.

## 2. Покупки після 18:00 (вечірній період)

Аналіз показує чітку закономірність: **ранкові та вечірні вподобання покупців суттєво різняться**, і ці зміни повторюються в кожному місті, але з власною специфікою.

## Mandalay
- **Вранці:** найпопулярніша категорія — **Sports and travel 0.21**.  
- **Увечері:** ця категорія **просідає 0.14**, її місце займає **Fashion accessories 0.26**, яка не була лідером вранці **0.15**.

**Що це означає:**  
У першій половині дня покупці більше орієнтуються на практичні або активні товари, а ввечері — на імпульсивні або стильові покупки.

## Naypyitaw
- **Вранці:** лідирує **Fashion accessories 0.21**.  
- **Увечері:** попит на неї зменшується **0.17**, і на перше місце виходить **Food and beverages 0.30**.

**Що це означає:**  
Покупці міста роблять більше fashion-покупок до 18:00, але після роботи чи навчання зміщуються до категорії їжі та напоїв.

## Yangon
- **Вранці:** домінує категорія **Home and lifestyle 0.21**.  
- **Увечері:** вона просідає **0.12**, а лідером стає **Food and beverages 0.22**, яка не була популярною вранці **0.16**.

**Що це означає:**  
Мешканці починають день із покупок для дому, а ввечері зміщуються до базових та швидких покупок.

## Загальна тенденція по всіх містах

### Категорії, що зростають після 18:00
- **Food and beverages** — **стабільно зростає у всіх містах**, стаючи однією з найпопулярніших вечірніх категорій.
  
### Категорії, що популярні вранці
- **Home and lifestyle**  
- **Sports and travel**

Ці категорії зменшують свою частку піля 18:00, що вказує на більш “планові” ранкові покупки.



### Гіпотеза 3: Спосіб оплати впливає на розмір середнього чеку
**Питання:**  
Чи більший середній чек у покупців, які оплачують карткою або електронними методами, порівняно з готівкою?


In [43]:
df.groupby('Payment')['Total'].agg(
    mean='mean',
    min='min',
    max='max'
).round(2)

,mean,min,max
Payment,,,
Cash,326.18,10.68,1003.59
Credit card,324.01,12.69,1042.65
Ewallet,318.82,13.42,1034.46


Середній чек не є вищим у покупців, які платять карткою або електронними методами.
Навпаки — готівка має найбільший середній чек (хоча різниця невелика). 
Найбільший чек був оплачений за допомогою електронних методів